In [1]:
%cd ..

c:\Users\namtv40\Projects\prefecthq-external-ingestion\ingestions


In [ ]:
from dotenv import load_dotenv

load_dotenv(".env.minio")


True

In [3]:
import sys
import os

sys.path.insert(0, r"C:\Users\namtv40\Projects\prefecthq-external-ingestion\ingestions")

In [4]:
import os
import sys
import json
import time
import requests
import pandas as pd
import pyarrow as pa
from dateutil import parser
import pyarrow.parquet as pq
from datetime import datetime


In [5]:

if not os.path.exists("./tmp/data"):
    os.makedirs("./tmp/data")


In [6]:
from common.config import *
from common.http_util import *
from common.crawler_util import *
from common.ambari_util import *


def fetch_resource_name_freshwork(resource_name, records, **kwargs):
    print("Start crawl : ", resource_name)
    start_time = time.time()

    HDFS_BASE = "s3a://vcs-raw/finance-raw"
    # STATE_PATH = rc["state_path"]
    # BASE_URL = None
    # RESOURCE_URL = None
    # API_KEY_PATH = rc["api_key_path"]
    # API_COOKIE_PATH = rc["api_cookie_path"]
    # QUERY_PARAMS = None
    ENABLE_STATE =False
    HIVE_DB = "finance_raw"
    # crawl_mode = "modified_and_new"
    crawl_mode = kwargs.get("crawl_mode", "static")
    schema_local_path = None
    # result_json_key = "deleted_deals"

    print(resource_name)
    start_time = time.time()
    # =========================
    # MAIN
    # =========================
    if ENABLE_STATE:
        last_state = read_last_state(resource_name)
        print("Last state =", last_state)
    else:
        last_state = None

    if not records:
        print("No new data")
        out_of_data = True
        return True

    # =========================
    # Pandas → Parquet
    # =========================

    now = datetime.now()
    partition_path = "{}/{}".format(HDFS_BASE, resource_name)

    filename = "data_{}_{}{:02d}{:02d}_{}{:02d}{:02d}.parquet".format(
        resource_name, now.year, now.month, now.day, now.hour, now.minute, now.second
    )
    local_parquet = "./tmp/data/finance_raw/{}/{}".format(resource_name, filename)
    os.makedirs(os.path.dirname(local_parquet), exist_ok=True)

    # df.to_parquet(local_parquet,engine="pyarrow", compression="snappy", index=False)
    records = convert_json_add_ts_columns(records)

    schema_tm_path = "./resources/parquet_schema/finance_raw/{}.json".format(resource_name)
    schema = None
    if schema_local_path:
        schema = load_pyarrow_schema_from_json(schema_local_path)

    if os.path.exists(schema_tm_path):
        schema = load_pyarrow_schema_from_json(schema_tm_path)

    if not schema:
        schema = infer_schema_from_json(records)
        schema_json = save_pyarrow_type_to_json(schema)
        write_file_json(schema_tm_path, schema_json)

    records = convert_json_list_by_arrow_schema(records, schema)

    df = pd.DataFrame(records)
    data_table = pa.Table.from_pandas(df, schema=schema, preserve_index=False)

    pq.write_table(data_table, local_parquet, compression="snappy")

    if crawl_mode == "static":
        replace_hdfs_https("{}".format(partition_path), local_parquet)
    else:
        upload_hdfs_https("{}".format(partition_path), local_parquet)

    print("Uploaded parquet to", partition_path)

    # =========================
    # Generate SQL (TEXT ONLY)
    # =========================
    sql = gen_spark_create_table(
        schema=schema,
        db=HIVE_DB,
        table=resource_name,
        location="{}/{}".format(HDFS_BASE, resource_name),
    )

    # filename = "create_table_{}_{}{:02d}{:02d}.sql".format(
    #     resource_name,
    #     now.hour,
    #     now.minute,
    #     now.second
    # )

    filename = "create_table_{}.sql".format(resource_name)

    local_sql = "./tmp/data/finance_raw/{}/{}".format(resource_name, filename)
    os.makedirs(os.path.dirname(local_sql), exist_ok=True)

    with open(local_sql, "w") as f:
        f.write(sql)

    # upload_hdfs_https(
    #     "{}/{}".format(HDFS_BASE, resource_name),
    #     local_sql
    # )

    print("Uploaded SQL definition")

    return False


In [7]:
import pandas as pd
import re
from pathlib import Path
from collections import defaultdict

import re
from collections import defaultdict
def excel_col_name(idx: int) -> str:
    """Zero-based index → Excel column (A, B, ..., AA)"""
    name = ""
    while idx >= 0:
        idx, rem = divmod(idx, 26)
        name = chr(rem + ord("A")) + name
        idx -= 1
    return name


def snake_case(text: str) -> str:
    text = (
        str(text)
        .strip()
        .lower()
        .replace("\n", " ")
    )
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", "_", text)
    return text

def read_csv_and_normalize_columns(
    file_path: str,
    mapping: dict,
    header_row=0,
    drop_rows=0
) -> pd.DataFrame:
    # đọc raw, chưa set header
    df = pd.read_csv(file_path, header=None, encoding="utf-8", dtype=str)
    raw_headers = df.iloc[header_row]
    seen = defaultdict(int)
    new_columns = []

    for idx, col in enumerate(raw_headers):
        if pd.isna(col) or col == "":
            base = "nan"
        else:
            # 1️⃣ ưu tiên dictionary
            base = mapping.get(str(col).strip())

            # 2️⃣ fallback snake_case
            if base is None:
                base = snake_case(col)
                print(col)

        seen[base] += 1

        if seen[base] > 1:
            excel_col = excel_col_name(idx).lower()
            base = f"{base}__{excel_col}"

        new_columns.append(base)

    # gán columns sạch
    df.columns = new_columns

    # drop header rows
    df = df.iloc[drop_rows:].reset_index(drop=True)
    df = df.fillna("")

    return df

def read_folder_and_union_csv(
    folder_path: str,
    mapping: dict,
    header_row=0,
    drop_rows=0,
    encoding="utf-8"
):
    dfs = []
    folder = Path(folder_path)

    files = sorted(folder.glob("*.csv"))
    if not files:
        raise ValueError("❌ Không tìm thấy file CSV nào trong folder")

    for file in files:
        df = read_csv_and_normalize_columns(
            file_path=file,
            mapping=mapping,
            drop_rows=drop_rows,
            header_row=header_row
        )

        # trace file nguồn
        df["source_file"] = file.name
        dfs.append(df)

    return pd.concat(dfs, ignore_index=True)

In [8]:
import pandas as pd
import re
from collections import defaultdict
def excel_col_name(idx: int) -> str:
    """Zero-based index → Excel column (A, B, ..., AA)"""
    name = ""
    while idx >= 0:
        idx, rem = divmod(idx, 26)
        name = chr(rem + ord("A")) + name
        idx -= 1
    return name


def snake_case(text: str) -> str:
    text = (
        str(text)
        .strip()
        .lower()
        .replace("\n", " ")
    )
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", "_", text)
    return text

def read_excel_and_normalize_columns(
    file_path: str,
    mapping: dict,
    sheet_name=0,
    header_row=0,
    drop_rows=0
) -> pd.DataFrame:
    # đọc raw, chưa set header
    df = pd.read_excel(file_path, sheet_name=sheet_name, header=None, dtype=str)

    raw_headers = df.iloc[header_row]
    seen = defaultdict(int)
    new_columns = []

    for idx, col in enumerate(raw_headers):
        if pd.isna(col) or col == "":
            base = "nan"
        else:
            # 1️⃣ ưu tiên dictionary
            base = mapping.get(col)

            # 2️⃣ fallback snake_case
            if base is None:
                print(f"Not found for col = '{col}'")
                base = snake_case(col)

        seen[base] += 1

        if seen[base] > 1:
            excel_col = excel_col_name(idx).lower()
            base = f"{base}__{excel_col}"

        new_columns.append(base)

    # gán columns sạch
    df.columns = new_columns

    # drop header rows
    df = df.iloc[drop_rows:].reset_index(drop=True)

    return df



In [9]:
COLUMN_DICT_PLAN_FIN  = {
  "segment":"segment"
}

filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\taichinh\raw"
resource_name = "finance_plan"
df = read_folder_and_union_csv(
      filename,
    mapping=COLUMN_DICT_PLAN_FIN,
    header_row=0,
    drop_rows=1,
)
df["subgroup"] = ""
df.head()

# 6. Chuyển sang JSON
records = df.to_dict(orient="records")
with open(resource_name +".json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=4)

print(f"✅ Done: {resource_name}.json created")

fetch_resource_name_freshwork(resource_name, records, crawl_mode="static")

plan_date
company
plan_viettel_group
plan_must
plan_nice
✅ Done: finance_plan.json created
Start crawl :  finance_plan
finance_plan
Replace Upload  s3a://vcs-raw/finance-raw/finance_plan ./tmp/data/finance_raw/finance_plan/data_finance_plan_20260417_175729.parquet
bucket=vcs-raw , key=finance-raw/finance_plan
objects_to_delete=[{'Key': 'finance-raw/finance_plan/hive.finance_raw.finance_plan_2280eb5765f1465f89be1fc4c3ad31c9.parquet'}]
bucket=vcs-raw , key=finance-raw/finance_plan/data_finance_plan_20260417_175729.parquet
Uploaded parquet to s3a://vcs-raw/finance-raw/finance_plan
Uploaded SQL definition


False

In [10]:
COLUMN_DICT_PLAN_FIN = {
    "Tên nhóm KH": "segment",
    "Phân nhóm KH": "subgroup",
    "Tập đoàn": "plan_viettel_group",
    "Must": "plan_must",
    "Nice": "plan_nice",
    "Tháng": "plan_month",
    "Năm": "plan_year",
    "Ngày tháng": "plan_date",
    "Đơn vị": "currency_code",
}

filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\plan-kinh-doanh-2026\Kế hoạch Nhóm KH_2026.xlsx"
resource_name = "finance_plan"
df = pd.read_excel(filename, dtype=str)

# Remove whitespace ở header nếu có
df.columns = df.columns.str.strip()
# Rename
df = df.rename(columns=COLUMN_DICT_PLAN_FIN)

df["source_file"] = "plan-kinh-doanh-2026\Kế hoạch Nhóm KH_2026.xlsx"
df["company"] = "VCS"
df["subgroup"] = df["subgroup"].fillna("")


print(df.columns)
print(df.dtypes)
df.head()


Index(['segment', 'subgroup', 'plan_viettel_group', 'plan_must', 'plan_nice',
       'plan_month', 'plan_year', 'plan_date', 'currency_code', 'source_file',
       'company'],
      dtype='object')
segment               object
subgroup              object
plan_viettel_group    object
plan_must             object
plan_nice             object
plan_month            object
plan_year             object
plan_date             object
currency_code         object
source_file           object
company               object
dtype: object


,segment,subgroup,plan_viettel_group,plan_must,plan_nice,plan_month,plan_year,plan_date,currency_code,source_file,company
0,DT BQP,,0,0,0,1,2026,2026-01-31 00:00:00,VNĐ,plan-kinh-doanh-2026\Kế hoạch Nhóm KH_2026.xlsx,VCS
1,DT nội bộ,,18846852007.24444,18846852007.24444,22616222408.693325,1,2026,2026-01-31 00:00:00,VNĐ,plan-kinh-doanh-2026\Kế hoạch Nhóm KH_2026.xlsx,VCS
2,DT ngoài trong nước,,7197948441.658479,8000000000,9600000000,1,2026,2026-01-31 00:00:00,VNĐ,plan-kinh-doanh-2026\Kế hoạch Nhóm KH_2026.xlsx,VCS
3,DT quốc tế,Thị trường,0,0,600000000,1,2026,2026-01-31 00:00:00,VNĐ,plan-kinh-doanh-2026\Kế hoạch Nhóm KH_2026.xlsx,VCS
4,DT quốc tế,Quốc tế,125000000,1200000000,1600000000,1,2026,2026-01-31 00:00:00,VNĐ,plan-kinh-doanh-2026\Kế hoạch Nhóm KH_2026.xlsx,VCS


In [11]:
# 6. Chuyển sang JSON
records = df.to_dict(orient="records")
with open(resource_name +".json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=4)

print(f"✅ Done: {resource_name}.json created")

fetch_resource_name_freshwork(resource_name, records, crawl_mode="modified_and_new")

✅ Done: finance_plan.json created
Start crawl :  finance_plan
finance_plan
Add Upload  s3a://vcs-raw/finance-raw/finance_plan ./tmp/data/finance_raw/finance_plan/data_finance_plan_20260417_175736.parquet
bucket=vcs-raw , key=finance-raw/finance_plan/data_finance_plan_20260417_175736.parquet
Uploaded parquet to s3a://vcs-raw/finance-raw/finance_plan
Uploaded SQL definition


False

--------------------

In [12]:
resource_name = "product_revenue_plan"
filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\plan_revenue\RAW_Dieu chinh Tach KH tung thang_N2025.xlsx"

import pandas as pd
import re

# ==============================
# 1️⃣ READ FILE
# ==============================
df = pd.read_excel(filename,sheet_name="KH N2025 (theo nhóm SPDV)", header=[0,1,2])

# Flatten header
df.columns = [
    "_".join([str(x).strip() for x in col if pd.notna(x)])
    for col in df.columns
]

# Rename 3 cột đầu
df = df.rename(columns={
    df.columns[0]: "tt",
    df.columns[1]: "ma_spdv",
    df.columns[2]: "ten_spdv",
})

# ==============================
# 2️⃣ IDENTIFY GROUP ROW
# ==============================
def is_roman(val):
    if pd.isna(val):
        return False
    return bool(re.fullmatch(r"[IVXLCDM]+", str(val).strip().upper()))

df["is_group"] = (
    df["ma_spdv"].isna() |
    df["ma_spdv"].astype(str).str.strip().eq("") |
    df["tt"].apply(is_roman)
)

# ==============================
# 3️⃣ CREATE PRODUCT GROUP
# ==============================
df["product_group"] = df["ten_spdv"].where(df["is_group"])
df["product_group"] = df["product_group"].ffill()

# ==============================
# 4️⃣ REMOVE GROUP ROWS
# ==============================
df_detail = df[~df["is_group"]].copy()

# Rename columns
df_detail = df_detail.rename(columns={
    "tt": "row_no",
    "ma_spdv": "product_code",
    "ten_spdv": "product_name",
})

# ==============================
# 5️⃣ MELT MONTH COLUMNS
# ==============================
value_cols = [
    c for c in df_detail.columns
    if "KH T" in c and ("MUST" in c.upper() or "NICE" in c.upper())
]

df_long = df_detail.melt(
    id_vars=["row_no", "product_code", "product_name", "product_group"],
    value_vars=value_cols,
    var_name="raw_column",
    value_name="target_value"
)

# ==============================
# 6️⃣ PARSE PERIOD & TARGET TYPE
# ==============================
def parse_column(col):
    # Extract month/year
    m = re.search(r"T(\d+)/(\d+)", col)
    if not m:
        return pd.Series([None, None, None, None])

    month = int(m.group(1))
    yy = m.group(2)
    year = int("20" + yy) if len(yy) == 2 else int(yy)

    target_type = "must" if "MUST" in col.upper() else "nice"

    period = f"{year}-{month:02d}"

    return pd.Series([period, year, month, target_type])

df_long[["period", "year", "month", "target_type"]] = df_long["raw_column"].apply(parse_column)

# ==============================
# 7️⃣ CLEAN VALUE + FILL NaN = 0
# ==============================
df_long["target_value"] = (
    df_long["target_value"]
        .astype(str)
        .str.replace(",", "", regex=False)
)

df_long["target_value"] = (
    pd.to_numeric(df_long["target_value"], errors="coerce")
      .fillna(0)
      .mul(1_000_000)   # nhân 10^6
      .astype("int64")   # BIGINT
)

# ==============================
# 8️⃣ FINAL CLEANUP
# ==============================
df_long = df_long.drop(columns=["raw_column"])

df_long = df_long.sort_values(
    ["product_group", "product_code", "year", "month", "target_type"]
)
df_long["currency_code"] = "VND"
import pandas as pd

df_long["snapshot_at"] = pd.Timestamp.now(tz="Asia/Ho_Chi_Minh").strftime("%Y-%m-%d %H:%M:%S")
df_long["snapshot_version"] = 1
df_long = df_long.reset_index(drop=True)

# ==============================
# 9️⃣ RESULT
# ==============================
# Pivot target_type thành cột
df_wide = (
    df_long
        .pivot_table(
            index=[
                "row_no",
                "product_group",
                "product_code",
                "product_name",
                "year",
                "month",
                "period",
                "currency_code",
                "snapshot_at",
                "snapshot_version",
            ],
            columns="target_type",
            values="target_value",
            aggfunc="sum",
            fill_value=0   # Quan trọng
        )
        .reset_index()
)

df_wide.columns.name = None

# Đảm bảo có đủ 2 cột
for col in ["must", "nice"]:
    if col not in df_wide.columns:
        df_wide[col] = 0

df_wide["must"] = df_wide["must"].astype("int64")
df_wide["nice"] = df_wide["nice"].astype("int64")

# Sắp xếp
df_wide = df_wide.sort_values(
    ["product_group", "product_code", "year", "month"]
)

df_wide.head(20)


,row_no,product_group,product_code,product_name,year,month,period,currency_code,snapshot_at,snapshot_version,must,nice
0,1,"HST SOC (Sản phẩm, dịch vụ thuộc HST Soc)",VCS001,MSS,2025,1,2025-01,VND,2026-04-17 17:57:54,1,10967036264,12972906453
1,1,"HST SOC (Sản phẩm, dịch vụ thuộc HST Soc)",VCS001,MSS,2025,2,2025-02,VND,2026-04-17 17:57:54,1,11478834658,13554673435
2,1,"HST SOC (Sản phẩm, dịch vụ thuộc HST Soc)",VCS001,MSS,2025,3,2025-03,VND,2026-04-17 17:57:54,1,21081028689,24838547180
3,1,"HST SOC (Sản phẩm, dịch vụ thuộc HST Soc)",VCS001,MSS,2025,4,2025-04,VND,2026-04-17 17:57:54,1,8646618351,9581995114
4,1,"HST SOC (Sản phẩm, dịch vụ thuộc HST Soc)",VCS001,MSS,2025,5,2025-05,VND,2026-04-17 17:57:54,1,12121405196,13395154438
5,1,"HST SOC (Sản phẩm, dịch vụ thuộc HST Soc)",VCS001,MSS,2025,6,2025-06,VND,2026-04-17 17:57:54,1,21925147421,25713911076
6,1,"HST SOC (Sản phẩm, dịch vụ thuộc HST Soc)",VCS001,MSS,2025,7,2025-07,VND,2026-04-17 17:57:54,1,11606564179,12146342449
7,1,"HST SOC (Sản phẩm, dịch vụ thuộc HST Soc)",VCS001,MSS,2025,8,2025-08,VND,2026-04-17 17:57:54,1,14283955956,14997575481
8,1,"HST SOC (Sản phẩm, dịch vụ thuộc HST Soc)",VCS001,MSS,2025,9,2025-09,VND,2026-04-17 17:57:54,1,21155561740,24196563369
9,1,"HST SOC (Sản phẩm, dịch vụ thuộc HST Soc)",VCS001,MSS,2025,10,2025-10,VND,2026-04-17 17:57:54,1,12798937900,19701470540


In [13]:
print("Số tháng:", df_long["month"].nunique())
print("Danh sách tháng:", sorted(df_long["month"].unique()))

Số tháng: 12
Danh sách tháng: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]


In [14]:
# 6. Chuyển sang JSON

resource_name = "product_revenue_plan"
records = df_wide.to_dict(orient="records")
with open(resource_name +".json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=4)

print(f"✅ Done: {resource_name}.json created")

fetch_resource_name_freshwork(resource_name, records, crawl_mode="static")

✅ Done: product_revenue_plan.json created
Start crawl :  product_revenue_plan
product_revenue_plan
Replace Upload  s3a://vcs-raw/finance-raw/product_revenue_plan ./tmp/data/finance_raw/product_revenue_plan/data_product_revenue_plan_20260417_175755.parquet
bucket=vcs-raw , key=finance-raw/product_revenue_plan
objects_to_delete=[{'Key': 'finance-raw/product_revenue_plan/hive.finance_raw.product_revenue_plan_f2895dca214b44d99413228ba8df01a3.parquet'}, {'Key': 'finance-raw/product_revenue_plan_2026/hive.finance_raw.product_revenue_plan_2026_2ac64ec55b4a439f9b3e2b0a59d3f4ac.parquet'}]
bucket=vcs-raw , key=finance-raw/product_revenue_plan/data_product_revenue_plan_20260417_175755.parquet
Uploaded parquet to s3a://vcs-raw/finance-raw/product_revenue_plan
Uploaded SQL definition


False

In [15]:
CUSTOMER_PRODUCT_MAPPING = {
    # ===== Customer group =====
    "Tên nhóm KH": "customer_group_name",
    "Phân nhóm KH": "customer_subgroup_name",
    "Nhóm": "group_name",
    "Phân nhóm": "subgroup_name",

    # ===== Product / Service =====
    "Mã SPDV": "service_code",
    "Sản phẩm": "service_name",
    "Đơn vị tính": "amount_unit",

    # ===== Priority / Importance =====
    "Must": "priority_must",
    "Should": "priority_should",
    "Nice": "priority_nice",

    # ===== Time =====
    "Tháng": "report_month",
    "Năm": "report_year",
    "Ngày tháng": "report_date",
}

filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\kinhdoanh-03-18-2026-18-57-47_files_list\Plan_SPDV_2026.xlsx"

df2 = read_excel_and_normalize_columns(
      filename,
    sheet_name="product_revenue_plan",
    mapping=CUSTOMER_PRODUCT_MAPPING,
    header_row=0,
    drop_rows=1,
)
df2["source_file"] = filename.split("\\")[-1]
df = df2.fillna("")

resource_name = "product_revenue_plan_2026"

df.head()
print(len(df))

# 6. Chuyển sang JSON
records = df.to_dict(orient="records")
with open(resource_name +".json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=4)

print(f"✅ Done: {resource_name}.json created")


1584
✅ Done: product_revenue_plan_2026.json created


In [16]:
resource_name = "product_revenue_plan_2026"
fetch_resource_name_freshwork(resource_name, records, crawl_mode="static")

Start crawl :  product_revenue_plan_2026
product_revenue_plan_2026
Replace Upload  s3a://vcs-raw/finance-raw/product_revenue_plan_2026 ./tmp/data/finance_raw/product_revenue_plan_2026/data_product_revenue_plan_2026_20260417_175800.parquet
bucket=vcs-raw , key=finance-raw/product_revenue_plan_2026
objects_to_delete=[]
bucket=vcs-raw , key=finance-raw/product_revenue_plan_2026/data_product_revenue_plan_2026_20260417_175800.parquet
Uploaded parquet to s3a://vcs-raw/finance-raw/product_revenue_plan_2026
Uploaded SQL definition


False